# 03 — Integração e limpeza

**Objetivo:** juntar as três bases numa única tabela por município, e produzir
a versão limpa que o notebook 04 usa para testar a hipótese.

**Entradas:**
- `data/processed/censo_domicilios_sp.csv` (notebook 01 — IBGE Censo 2022)
- `data/processed/internacoes_sp.csv` (notebook 02 — SIH/SUS por residência)
- `data/external/idh_sp.csv` (Atlas Brasil — variável de controle)

**Saídas:**
- `dataset_municipios_sp_bruto.csv` — 645 municípios, sem filtros
- `dataset_municipios_sp.csv` — versão **limpa** (é a que o notebook 04 usa)

**Por que uma linha por município (e não por ano):** os exports do TabNet que
temos trazem o total do período, sem quebra anual (ver notebook 02). Isso
também evita [pseudorreplicação](https://en.wikipedia.org/wiki/Pseudoreplication):
`pct_idosos_sozinhos` vem do Censo 2022 e seria constante ao longo dos anos,
então repetir cada município 5 vezes inflaria o "n" artificialmente sem
acrescentar informação nenhuma sobre a hipótese.


In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path("..") / "src"))
import config  # noqa: E402

import pandas as pd
import numpy as np

censo = pd.read_csv(config.DATA_PROCESSED / "censo_domicilios_sp.csv")
internacoes = pd.read_csv(config.DATA_PROCESSED / "internacoes_sp.csv")

print("censo:", censo.shape, "| internacoes:", internacoes.shape)


## 3.1 IDH municipal (variável de controle)

O IDH entra como controle: municípios mais pobres tendem a ter mais
internações por razões que não têm nada a ver com morar sozinho. Sem esse
controle, qualquer associação encontrada poderia ser só um reflexo da
desigualdade socioeconômica entre municípios.

⚠️ O IDHM disponível é o de 2010 (o Censo 2022 ainda não tem IDHM oficial
publicado) e cobre 259 dos 645 municípios — declarar as duas coisas como
limitação no artigo.


In [ ]:
idh = pd.read_csv(config.DATA_EXTERNAL / "idh_sp.csv")
idh["municipio_norm"] = idh["municipio"].apply(config.normalizar_municipio)
idh = idh[["municipio_norm", "idh"]].rename(columns={"idh": "idhm"})
idh = idh.drop_duplicates(subset="municipio_norm")

print(f"{len(idh)} municípios no arquivo de IDH")
print(f"{idh['municipio_norm'].isin(censo['municipio_norm']).sum()} deles batem com o Censo")


## 3.2 Base por município (bruta, 645 municípios)

O merge parte da lista completa de municípios do **Censo** (645), não das
internações — assim, municípios sem nenhuma internação em algum capítulo
entram com 0 em vez de sumirem da base.


In [ ]:
causas = list(config.CAUSAS_SIH.keys())

# uma coluna por capítulo CID-10
internacoes_wide = internacoes.pivot_table(
    index="municipio_norm", columns="causa", values="internacoes", aggfunc="sum"
).reset_index()

mun = (
    censo
    .merge(internacoes_wide, on="municipio_norm", how="left")
    .merge(idh, on="municipio_norm", how="left")
)
mun[causas] = mun[causas].fillna(0).astype(int)

mun["internacoes_total"] = mun[causas].sum(axis=1)
mun["taxa_internacao_100k_domicilios_idosos"] = (
    mun["internacoes_total"] / mun["domicilios_resp_idoso"] * 100_000
).round(2)

print(mun.shape)
print(f"Internações somadas: {mun['internacoes_total'].sum():,} (deve bater com o total do notebook 02)")
print(f"Municípios sem IDH: {mun['idhm'].isna().sum()}")
mun.head()


## 3.3 Limpeza

Três tratamentos, nesta ordem:

1. **Arredondamento** — as taxas ficam com 2 casas decimais (feito acima). Não
   há perda de informação real: a precisão adicional era só ruído de divisão.
2. **Outliers** — removidos pela regra de Tukey (taxa acima de Q3 + 1,5×IQR).
   Agora que o dado é por residência, os outliers restantes são
   predominantemente municípios **muito pequenos**, onde o denominador
   (domicílios com responsável idoso) é baixo e a taxa por 100 mil fica
   instável — é variância estatística legítima, não o viés sistemático que
   tínhamos antes com o dado por local de internação.
3. **Municípios sem IDH** — eliminados em vez de imputados. Cerca de 60% da
   coluna está ausente, e não há aqui fonte confiável para imputar com
   critério; inventar valor para a variável de controle seria pior do que
   reduzir a amostra.

A base bruta fica salva à parte para a checagem de robustez do notebook 04.


In [ ]:
q1, q3 = mun["taxa_internacao_100k_domicilios_idosos"].quantile([0.25, 0.75])
limite_tukey = q3 + 1.5 * (q3 - q1)
mun["outlier_taxa"] = mun["taxa_internacao_100k_domicilios_idosos"] > limite_tukey

print(f"Limite de Tukey (Q3 + 1,5xIQR): {limite_tukey:,.0f} internações por 100 mil")
print(f"Outliers detectados: {mun['outlier_taxa'].sum()}")
print()
print(mun[mun["outlier_taxa"]][["municipio", "domicilios_resp_idoso", "internacoes_total",
                                 "taxa_internacao_100k_domicilios_idosos"]]
      .sort_values("taxa_internacao_100k_domicilios_idosos", ascending=False)
      .to_string(index=False))


In [ ]:
mun.to_csv(config.DATA_PROCESSED / "dataset_municipios_sp_bruto.csv", index=False)

limpo = mun[~mun["outlier_taxa"] & mun["idhm"].notna()].drop(columns="outlier_taxa")
limpo.to_csv(config.DATA_PROCESSED / "dataset_municipios_sp.csv", index=False)

print(f"Bruto: {len(mun)} municípios  -> dataset_municipios_sp_bruto.csv")
print(f"Limpo: {len(limpo)} municípios -> dataset_municipios_sp.csv")
print(f"  removidos por outlier: {mun['outlier_taxa'].sum()}")
print(f"  removidos por falta de IDH: {mun['idhm'].isna().sum()}")


## 3.4 Conferência — Rio Claro

Rio Claro é o município do estudo de caso (notebook 05), então vale conferir
que ele sobreviveu à limpeza e que os números fazem sentido.


In [ ]:
rc = limpo[limpo["municipio"] == config.RIO_CLARO_NOME]
if rc.empty:
    print(f"ATENÇÃO: {config.RIO_CLARO_NOME} caiu fora da base limpa -- confira o bruto.")
    rc = mun[mun["municipio"] == config.RIO_CLARO_NOME]
print(rc.T.to_string())
